[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/07_dataframes/07_3_Spotting_Problems.ipynb)

# 07.3: Spotting Problems in Data

So far we have mostly taken the data at face value. Real data has gaps and errors, and pandas will calculate an answer from it anyway, without any warning. This short notebook shows how to check a dataset for common problems and how to handle the simple ones. The goal here is to get in the habit of checking before you trust a result.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

This data comes from the Centers for Disease Control and has one row for each reported COVID-19 case in McLean County, Illinois, from March 2020 to March 2021. We will keep a few of the columns.

In [ ]:
import pandas as pd

covid = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_covid_mclean_county_retrieved_4_1_21.csv")
covid = covid[["case_month", "age_group", "sex", "race", "hosp_yn"]]
covid.head()

## Finding missing values

`.isna().sum()` on a DataFrame counts the missing values in each column. This is the same `.isna()` you saw in 06.3 for a series.

In [ ]:
print(len(covid))
covid.isna().sum()

There are 15334 rows. `age_group` has 134 missing values, `sex` has 679 and `race` has 1829. `case_month` and `hosp_yn` appear to be complete.

`.notna()` is the opposite of `.isna()`, so `.notna().sum()` counts the values that are present. `age_group` has 15200, which is 15334 minus the 134 missing values.

In [ ]:
covid["age_group"].notna().sum()

## Missing data does not always look missing

`.value_counts()` skips missing values unless you pass `dropna=False`, which also gives you a look at every value in a column.

In [ ]:
covid["sex"].value_counts(dropna=False)

Unknown sex is recorded in three different ways: 679 are `NaN`, 186 say `Unknown` and 44 say `Missing`. pandas counts only the `NaN` values as missing. The other two are ordinary strings to it. The problem is much larger in `hosp_yn`, which records whether the person was hospitalized.

In [ ]:
covid["hosp_yn"].value_counts(dropna=False)

`.isna().sum()` reported nothing wrong with this column, yet 10789 rows say `Missing` and 1109 say `Unknown`. Only 3436 of the 15334 rows have a real answer. If you computed the percentage of cases that were hospitalized using all the rows, you would get 424 out of 15334, or about 3 percent. Using only the rows with an answer gives 424 out of 3436, or about 12 percent. So look at the values in a column, and do not rely on `.isna()` alone.

## What to do about it: dropping and filling

`.dropna()` removes rows with missing values. By default it drops a row if **any** column is missing, which can lose far more than you intended. The `subset=` argument limits it to the columns you name.

In [ ]:
print(len(covid.dropna()))
print(len(covid.dropna(subset=["age_group"])))

Dropping on every column leaves 13505 rows, while dropping only where `age_group` is missing leaves 15200, which loses just the 134 rows we expected. `.fillna()` replaces missing values with one you choose. That can be a good way to keep a category of unknowns visible in your counts. Both of these return a new result, and `covid` does not change unless you assign it back.

In [ ]:
covid["age_group"].fillna("Unknown").value_counts()

The 134 missing values now appear as `Unknown`, so they stay visible in the counts. For a numeric column, a common choice is to fill with the mean or the median of the values that exist. This keeps the row, but it also makes up a value, so use it with care. `.dropna()` also works on a single series, and `.dropna(axis="columns")` drops whole columns instead of rows.

In [ ]:
print(covid.dropna(axis="columns").columns.to_list())
print(len(covid["sex"].dropna()))

Only `case_month` and `hosp_yn` have no missing values, so they are the only columns left. Dropping missing values from `sex` leaves 14655 of the 15334 values.

Rows that say `Missing` or `Unknown` are ordinary strings, so you remove them with a mask, the way you learned in 07.2.

In [ ]:
answered = covid[covid["hosp_yn"].isin(["Yes", "No"])]
print(len(answered))
(answered["hosp_yn"] == "Yes").mean()

This confirms the 3436 rows and a share of about 0.123. Whichever choice you make, it is good practice to report how many rows you left out, since the people with an answer may differ from the people without one.

## Duplicate rows

Rows are sometimes repeated, for example when someone submits a form twice. `.duplicated()` makes a mask that is `True` for each row that repeats an earlier row, and `.drop_duplicates()` removes those rows. The `subset=` argument compares only the columns you name.

In [ ]:
survey = pd.DataFrame({"student": ["Ana", "Ben", "Ana", "Cal", "Ben"],
                       "major": ["Math", "Art", "Math", "Art", "History"]})
print(survey.duplicated().sum())
print(len(survey.drop_duplicates()))
print(len(survey.drop_duplicates(subset=["student"])))

One row is an exact repeat, Ana in Math, so dropping duplicates leaves 4 rows. Comparing only `student` leaves 3, because Ben's second row counts as a repeat even though its major differs, and the first row for each student is the one kept.

A repeated row is not always a mistake. In the COVID table, 14655 rows repeat an earlier row in the five columns we kept. They are different people who happen to share a month, age group, sex, race and hospital status, and removing them would throw away nearly all of the data. So before you drop duplicates, ask whether two identical rows could really be two different things.

In [ ]:
covid.duplicated().sum()

## Values that cannot be right

Some problems have nothing to do with missing values. In 07.1, the Broadway `Capacity` column had a max of 892. A mask makes it easy to see how many rows look wrong.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
print((df["Capacity"] > 100).sum())
print((df["Gross Potential"] == 0).sum())
df["Capacity"].sort_values(ascending=False).head(3)

`Capacity` is above 100 in 2129 rows. Nearly all of those, 2117 of them, are at 110 or below, which is believable because a theatre can sell some standing room. The other 12 go as high as 892, which is not believable. `Gross Potential` is 0 in 2129 rows as well (the matching count is a coincidence, since they are mostly different rows). Those zeros are mostly from the mid-1990s and appear alongside real sales, so a 0 here most likely means the value was not recorded, even though it looks like a real number. Nothing in the data tells you this directly. You find it by comparing the numbers with what you know about the world.

## Numbers stored as text

If one value in a column is not a number, the whole column is treated as text, and math on it will fail. `pd.to_numeric()` converts a series to numbers. With `errors="coerce"`, anything it cannot convert becomes `NaN`, and then the statistics you already know will skip it.

In [ ]:
ages = pd.Series(["34", "27", "Unknown", "51"])
pd.to_numeric(ages, errors="coerce")

`34`, `27` and `51` are now numbers, and `Unknown` has become `NaN`.

Answers typed by people often need some text cleaning before they can be converted. This short example uses the `.str` methods from 06.4. It lowers the case and strips spaces, keeps the first number from a range such as `4-5`, turns a word into a digit, and only then converts.

In [ ]:
answers = pd.Series(["4", " 4-5", "One", "7", "not sure"])
cleaned = answers.str.lower().str.strip()
cleaned = cleaned.str.split("-").str[0]
cleaned = cleaned.str.replace("one", "1")
pd.to_numeric(cleaned, errors="coerce")

The result is 4, 4, 1, 7 and `NaN`, with `not sure` left as missing. Going the other way, `.astype()` converts a column to a type you name, so `df["Year"].astype(str)` would turn the years into text. That is useful when you want to treat a number such as a ZIP code as a label.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

raw_survey = pd.DataFrame({"name": ["Ana", "Ben", "Cal", "Ben", "Dee", "Eli"],
                           "age": ["19", "20", "unknown", "20", "21", None],
                           "color": [" Red", "blue", None, "blue", "RED ", "Blue"]})
survey = raw_survey.copy()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
Create a series named `missing_counts` that gives the number of missing values in each column of `survey`.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "missing_counts", raw_survey.isna().sum(), "Use .isna() and then .sum() on the whole table.")

### Problem 2
Create a DataFrame named `has_color` with only the rows that have a value in `color`. Rows missing only `age` should stay.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "has_color", raw_survey.dropna(subset=["color"]), "Use .dropna() with subset=.")

### Problem 3
Create a series named `color_filled` from the `color` column in which missing values are replaced with `unknown`.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "color_filled", raw_survey["color"].fillna("unknown"), "Use .fillna() on the column.")

### Problem 4
Create a series named `clean_color` from the `color` column with the spaces removed from the ends and everything in lower case.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "clean_color", raw_survey["color"].str.strip().str.lower(), "Chain .str.strip() and .str.lower().")

### Problem 5
Create a series named `age_numbers` by converting the `age` column to numbers, with anything that cannot be converted becoming `NaN`.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "age_numbers", pd.to_numeric(raw_survey["age"], errors="coerce"), "Use pd.to_numeric() with errors=\"coerce\".")

### Problem 6
Store the number of rows that repeat an earlier row in a variable named `repeat_count`.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "repeat_count", raw_survey.duplicated().sum(), "Use .duplicated() and then .sum().")

### Problem 7
Create a DataFrame named `no_repeats` that is `survey` with the repeated rows removed. `survey` should not change.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "no_repeats", raw_survey.drop_duplicates(), "Use .drop_duplicates() and assign the result to no_repeats.", ("survey", raw_survey))

## What's next

You can now check a dataset for gaps, repeats and impossible values before you use it. Once you trust the data, the next question is usually a comparison, such as whether musicals earn more than plays. Notebook 07.4 shows how to compare groups.